# Lab 3.2 &mdash; Perception &mdash; Turning Raw Output into an Observation

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Turn an opaque record into something the model cannot misread
- Distinguish the four kinds of nothing a tool can return
- Stamp freshness, completeness and authority onto every observation
- Measure the token cost of being clear -- it is usually negative

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The cheapest accuracy in the course.** No model change, no extra call: you are only
> deciding what the agent gets to see.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

A tool returns data. An **observation** is data plus the meaning the agent needs to act on it. Most
agent failures blamed on reasoning are really the model guessing at a field it was never told how
to read &mdash; and guessing the reassuring way.

## Section 1 &mdash; Decode the opaque record

The upstream system speaks in codes. The agent should never have to guess what they mean.

In [ ]:
# what the upstream ledger actually returns
RAW = {"i": 1003, "a": 990000.0, "c": "USD", "s": 2, "rc": 7,
       "vd": "2026-09-02", "cp": 19, "f": None, "x": [], "ts": 1757030400}

STATUS_CODES = {0: "settled", 1: "failed", 2: "held"}
REASON_CODES = {5: "INSUFFICIENT_FUNDS", 6: "INVALID_IBAN", 7: "LIMIT_BREACH", 8: "SANCTIONS_REVIEW"}
LIMIT_USD = 500000.0

def observe(raw: dict) -> str:
    """Render one raw ledger record as an observation a model can act on.

    Must state: the reference, the decoded status, the decoded reason, and -- where the
    reason is a limit breach -- the comparison that explains it.
    """
    status = STATUS_CODES.get(raw["s"], "unknown")
    reason = REASON_CODES.get(raw["rc"], "unknown")
    lines = [f"PMT-{raw['i']} is {status.upper()}.", f"Reason: {reason}"]
    if reason == "LIMIT_BREACH":
        lines.append(BLANK)          # TODO: the comparison that makes the reason self-evident,
                                     # e.g. "USD 990,000.00 exceeds the 500,000.00 limit."
    return " ".join(lines)

In [ ]:
# --- Self-check: Section 1
check("the reference is stated in the form the tools use",
      lambda: "PMT-1003" in observe(RAW))
check("the status code is decoded, not passed through",
      lambda: "HELD" in observe(RAW) and '"s": 2' not in observe(RAW))
check("the reason code is decoded", lambda: "LIMIT_BREACH" in observe(RAW))
check("the limit comparison is spelled out",
      lambda: "990,000.00" in observe(RAW) and "500,000.00" in observe(RAW),
      "the model should not have to know the limit to understand the reason")
check("a settled record needs no comparison",
      lambda: "exceeds" not in observe({**RAW, "s": 0, "rc": 0}))
check("the observation is far shorter than the raw record",
      lambda: len(observe(RAW)) < len(json.dumps(RAW)) * 1.5)

## Section 2 &mdash; The four kinds of nothing

An empty result is the most dangerous thing a tool returns, because the reassuring reading and the
alarming one look identical.

In [ ]:
def describe_empty(result: dict) -> str:
    """Say which kind of nothing this is.

    result carries: ran (bool), error (str|None), matches (list), total (int|None)
    Returns one of: "checked_clear", "not_run", "partial", "unknown_total"
    """
    if result["error"] or not result["ran"]:
        return BLANK                 # TODO: the check did not actually happen
    if result["total"] is None:
        return "unknown_total"
    if len(result["matches"]) < result["total"]:
        return "partial"
    return "checked_clear"

def render_screening(result: dict) -> str:
    """The observation a screening tool should return -- never a bare empty list."""
    kind = describe_empty(result)
    return {
        "checked_clear":  f"Sanctions screening ran and found 0 of {result['total']} records matching. Clear.",
        "not_run":        f"Sanctions screening DID NOT RUN ({result['error'] or 'no reason given'}). Result unknown -- do not treat as clear.",
        "partial":        f"Sanctions screening returned {len(result['matches'])} of {result['total']} records. Incomplete.",
        "unknown_total":  "Sanctions screening returned results but the total is unknown. Completeness cannot be established.",
    }[kind]

In [ ]:
# --- Self-check: Section 2
CLEAR   = {"ran": True,  "error": None,        "matches": [], "total": 0}
TIMEOUT = {"ran": False, "error": "timeout",   "matches": [], "total": None}
PARTIAL = {"ran": True,  "error": None,        "matches": [], "total": 47}
NOTOTAL = {"ran": True,  "error": None,        "matches": [], "total": None}

check("a genuine all-clear is reported as clear",
      lambda: describe_empty(CLEAR) == "checked_clear")
check("a check that did not run is NOT reported as clear",
      lambda: describe_empty(TIMEOUT) == "not_run",
      "this is the distinction the whole section exists for")
check("an incomplete result is flagged", lambda: describe_empty(PARTIAL) == "partial")
check("an unknown total is flagged", lambda: describe_empty(NOTOTAL) == "unknown_total")
check("the not-run observation warns against the reassuring reading",
      lambda: "do not treat as clear" in render_screening(TIMEOUT).lower())
check("all four render to different text",
      lambda: len({render_screening(r) for r in (CLEAR, TIMEOUT, PARTIAL, NOTOTAL)}) == 4)

for name, r in (("clear", CLEAR), ("timeout", TIMEOUT), ("partial", PARTIAL), ("no total", NOTOTAL)):
    try:
        print(f"  {name:9} -> {render_screening(r)}")
    except NameError:
        print("(fill in describe_empty above)"); break

## Section 3 &mdash; Stamp what the agent cannot see

Freshness, completeness and authority are absent from most tool results, and the agent assumes the
convenient value for each.

In [ ]:
NOW = 1757030400 + 7200              # pretend "now" is two hours after the record's timestamp

def stamp(observation: str, *, as_of: int, now: int = NOW,
          shown: int = 1, total: int = 1, binding: bool = True) -> str:
    """Append the three things a raw result never carries."""
    age_min = (now - as_of) // 60
    freshness = "live" if age_min < 5 else f"as of {age_min} minutes ago"
    parts = [observation, f"[{freshness}"]
    if shown < total:
        parts.append(f"; showing {shown} of {total}")
    if BLANK:                        # TODO: when should the observation warn it is not binding?
        parts.append("; DRAFT policy, not binding")
    return "".join(parts) + "]"

In [ ]:
# --- Self-check: Section 3
check("a stale record says how stale",
      lambda: "120 minutes ago" in stamp("x", as_of=1757030400))
check("a fresh record says live",
      lambda: "live" in stamp("x", as_of=NOW))
check("a truncated result says so",
      lambda: "showing 3 of 47" in stamp("x", as_of=NOW, shown=3, total=47))
check("a complete result does not add noise",
      lambda: "showing" not in stamp("x", as_of=NOW, shown=1, total=1))
check("a draft policy is marked as not binding",
      lambda: "not binding" in stamp("x", as_of=NOW, binding=False),
      "a draft and a ratified policy are both just text otherwise")
check("a binding policy carries no draft warning",
      lambda: "not binding" not in stamp("x", as_of=NOW, binding=True))

try:
    print("  " + stamp(observe(RAW), as_of=1757030400, shown=3, total=47, binding=False))
except NameError:
    print("(fill in the blanks above)")

## Run it for real

The same question, twice: once with the raw record, once with your observation. Same model, same
prompt &mdash; only what the agent can see has changed.

In [ ]:
QUESTION = ("Given the payment data below, say in one line who must action this and whether it may "
            "be released. Answer only from the data given.")

if llm_ready():
    try:
        print("--- with the raw record ---")
        print("  " + ask(f"{QUESTION}\n\nDATA: {json.dumps(RAW)}").strip()[:300])
        print("\n--- with your observation ---")
        obs = stamp(observe(RAW), as_of=1757030400, shown=1, total=1, binding=True)
        print("  " + ask(f"{QUESTION}\n\nDATA: {obs}").strip()[:300])
        print(f"\nraw record: {len(json.dumps(RAW))} chars   observation: {len(obs)} chars")
    except NameError:
        print("(fill in the blanks above, then re-run this cell)")

### Read it

Two things to check. First, whether the raw arm invented a meaning for `s: 2` or `rc: 7` &mdash; it has
no way to know them, so anything it says about status is a guess dressed as an answer.

Second, the character counts. The observation is usually **shorter** than the raw record as well as
clearer, because you dropped the fields nobody needed. Perception is one of the few places where
the cheap option and the accurate option are the same option.

In [ ]:
score()

## Your turn

1. `observe()` hardcodes `LIMIT_USD`. What happens when the limit is per-currency or per-client?
   Where should that number live so the observation stays truthful?
2. Add a fifth kind of nothing: the check ran, matched, but the results were **suppressed** by
   entitlements. How should that observation read so the agent neither ignores it nor over-reacts?